# 128 — Paralelismo, fan-out y map-reduce

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.**
map = 12 × (8 000×3e-6 + 600×15e-6) = 12 × (0.024 + 0.009) = 12 × 0.033 = **0.396 USD**.
reduce: entrada = 500 + 12×600 = 7 700 → 7 700×3e-6 = 0.0231; salida = 1 500×15e-6 =
0.0225 → **0.0456 USD**. Total = **0.4416 USD**. El map domina (~90 %): con partes
disjuntas, el sobrecoste del paralelismo es esencialmente el reduce.

**Ejercicio 2.** Fan-out = max(48) + 20 = **68 s**; secuencial = 127 + 20 = **147 s**;
speedup ≈ **2.16×** — decepcionante: el rezagado de 48 s se come el paralelismo.
Con timeout 20 s: fan-out = 20 + 20 = **40 s** (7/8 partes, marcado en limitations),
speedup ≈ 3.7×. La latencia del fan-out la fija el `max`, y controlar la cola de la
distribución (timeouts) vale más que añadir workers.

**Ejercicio 3.** n_max ≈ (200 000 − 500) / 600 ≈ **332 workers** en reduce plano.
Para n = 2 000: ⌈log₂ 2 000⌉ = **11 niveles** de reduce binario (cada nivel fusiona
pares, con entradas siempre acotadas).

**Ejercicio 4.** Los tres contratos comparten claves `{agent, score, finding}`
(verificado con asserts). El "reduce" del laboratorio es la consolidación del
supervisor: promedio informativo + decisión por mínimo — una fusión, no una
concatenación.


In [ ]:
result = run_lab("multiagent", seed=128)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
import math
P_IN, P_OUT = 3 / 1e6, 15 / 1e6

# Ejercicio 1
coste_map = 12 * (8000 * P_IN + 600 * P_OUT)
coste_reduce = (500 + 12 * 600) * P_IN + 1500 * P_OUT
coste_total = coste_map + coste_reduce
print(f"map={coste_map:.4f} reduce={coste_reduce:.4f} total={coste_total:.4f} USD")

# Ejercicio 2
lats = [12, 9, 15, 11, 48, 10, 13, 9]
lat_fanout = max(lats) + 20
lat_secuencial = sum(lats) + 20
speedup = lat_secuencial / lat_fanout
vivos = [l for l in lats if l <= 20]
# el coordinador espera hasta agotar el timeout (20 s) antes de degradar 7/8
lat_con_timeout = 20 + 20
print(f"fanout={lat_fanout}s secuencial={lat_secuencial}s speedup={speedup:.2f}x "
      f"| con timeout: {lat_con_timeout}s ({len(vivos)}/8 partes)")

# Ejercicio 3
n_max_plano = (200_000 - 500) // 600
niveles_arbol = math.ceil(math.log2(2000))
print(f"n_max_plano={n_max_plano} niveles_arbol={niveles_arbol}")

# Ejercicio 4
result = run_lab("multiagent", seed=128)
workers = result["result"]["workers"]
claves = [set(w) for w in workers]
assert all(c == {"agent", "score", "finding"} for c in claves)
print("map: 3 contratos homogéneos → reduce:", result["result"]["supervisor"])


## Reflexión

1. En el ejemplo, map cuesta 0.204 USD y reduce 0.031 USD. ¿Con qué crecimiento de n (o de t_out por worker) el reduce pasa a dominar el coste y qué harías entonces?
2. Los tres workers del laboratorio corren lógicamente en paralelo porque no comparten estado. ¿Qué cambio en la tarea (no en el código) rompería esa independencia y obligaría a abandonar map-reduce?
3. Si duplicar n reduce la latencia a la mitad pero duplica el coste, ¿qué dato de negocio necesitas para elegir n? Formula la decisión como una desigualdad.
